# Multi step workflow with Language Chain Expression Language

Create an AI Business Advisor that:

* Accepts an industry as input.
* Generates a business idea.
* Analyzes the strengths and weaknesses.
* Formats the results as a final report.

Use LCEL to chain prompts LLMs and output parsers.

In [1]:
from dotenv import load_dotenv

loaded = load_dotenv("../.env")
print("Loaded:", loaded)

Loaded: True


In [2]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough, RunnableParallel, RunnableLambda
from pydantic import BaseModel, Field
import os

In [3]:
# intantiate the LLM
llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0.0,
    api_key=os.getenv("OPENAI_API_KEY"),
    base_url="https://openai.vocareum.com/v1"
)

## Create a chain
This runnable should parse the provided input and also log it. So parsing and logging can run in parallel. Hence use RunnableParallel.

RunnableParallel, expects a map of steps. Below we have output and log but these names can be also different.

In [4]:
logs = []

In [5]:
parser = StrOutputParser()

In [6]:
parse_and_log_output_chain = RunnableParallel(
    output=parser,
    log=RunnableLambda(lambda x: logs.append(x))
)

## Idea Generation

Given industry LLM should generate some idea.

In [7]:
idea_prompt = PromptTemplate(
    template=("""You are an experienced business consultant.
    
Generate one innovative and practical business idea in the {industry} industry.

Provide a brief description of the idea.""")
)

In [8]:
idea_chain = idea_prompt | llm | parse_and_log_output_chain

In [9]:
idea_result = idea_chain.invoke("agro")

In [10]:
idea_result["output"]

'**Business Idea: Smart Vertical Farming Pods (SVFP)**\n\n**Description:**\nSmart Vertical Farming Pods (SVFP) are modular, self-sustaining farming units designed for urban environments, utilizing advanced technology to optimize space, resources, and crop yield. Each pod is equipped with hydroponic systems, LED grow lights, and IoT sensors that monitor and control environmental conditions such as temperature, humidity, and nutrient levels. \n\nThe SVFP can be installed in various locations, including rooftops, vacant lots, and even inside commercial buildings, making it accessible for urban dwellers and businesses looking to grow fresh produce locally. The pods are designed to be user-friendly, allowing individuals or small businesses to manage their crops through a mobile app that provides real-time data, growth tracking, and automated care reminders.\n\nAdditionally, the SVFP can incorporate a subscription model for maintenance and supply of seeds and nutrients, ensuring a steady rev

In [11]:
logs

[AIMessage(content='**Business Idea: Smart Vertical Farming Pods (SVFP)**\n\n**Description:**\nSmart Vertical Farming Pods (SVFP) are modular, self-sustaining farming units designed for urban environments, utilizing advanced technology to optimize space, resources, and crop yield. Each pod is equipped with hydroponic systems, LED grow lights, and IoT sensors that monitor and control environmental conditions such as temperature, humidity, and nutrient levels. \n\nThe SVFP can be installed in various locations, including rooftops, vacant lots, and even inside commercial buildings, making it accessible for urban dwellers and businesses looking to grow fresh produce locally. The pods are designed to be user-friendly, allowing individuals or small businesses to manage their crops through a mobile app that provides real-time data, growth tracking, and automated care reminders.\n\nAdditionally, the SVFP can incorporate a subscription model for maintenance and supply of seeds and nutrients, en

## Idea analysis
Now llm should analyze the generated output from idea generation step.


In [12]:
analysis_prompt = PromptTemplate(
    template=(
        """Please analyze the following idea: {idea}
        And create only 3 strengths and 3 weaknesses of the idea.
        """
    )
)

In [13]:
analysis_chain = analysis_prompt | llm | parse_and_log_output_chain

In [14]:
analysis_result = analysis_chain.invoke(idea_result["output"])

In [15]:
analysis_result["output"]

'### Strengths:\n\n1. **Space Optimization and Accessibility**: The modular design of Smart Vertical Farming Pods allows for efficient use of limited urban space. By utilizing rooftops, vacant lots, and indoor areas, the SVFP can bring fresh produce closer to consumers, reducing the need for transportation and making urban farming more accessible to city dwellers.\n\n2. **Technological Integration**: The incorporation of IoT sensors and a mobile app for monitoring and managing crops enhances user experience and efficiency. Real-time data on environmental conditions and automated care reminders can lead to higher crop yields and better resource management, appealing to both novice and experienced urban farmers.\n\n3. **Sustainability and Food Security**: SVFP addresses critical issues such as food security and sustainability by providing a local source of fresh, pesticide-free produce. This not only reduces the carbon footprint associated with transporting food but also promotes environ

In [16]:
for message in logs:
    print(message.content)
    print("-----")

**Business Idea: Smart Vertical Farming Pods (SVFP)**

**Description:**
Smart Vertical Farming Pods (SVFP) are modular, self-sustaining farming units designed for urban environments, utilizing advanced technology to optimize space, resources, and crop yield. Each pod is equipped with hydroponic systems, LED grow lights, and IoT sensors that monitor and control environmental conditions such as temperature, humidity, and nutrient levels. 

The SVFP can be installed in various locations, including rooftops, vacant lots, and even inside commercial buildings, making it accessible for urban dwellers and businesses looking to grow fresh produce locally. The pods are designed to be user-friendly, allowing individuals or small businesses to manage their crops through a mobile app that provides real-time data, growth tracking, and automated care reminders.

Additionally, the SVFP can incorporate a subscription model for maintenance and supply of seeds and nutrients, ensuring a steady revenue str

## Report

Create a business report from the strengts and weaknesses of the business idea.

In [17]:
report_prompt = PromptTemplate(
    template=(
        """Here is the business analysis:
        Strengths&Weaknesses: {output}
        
        Generate structured business report""" 
    )
)

In [18]:
class AnalysisReport(BaseModel):
    """Strengths and weaknesses about a business idea."""
    strengths: list = Field(default=[], description="Idea's strengths list")
    weaknesses: list = Field(default=[], description="Idea's weaknesses list")

In [19]:
report_chain = (report_prompt | llm.with_structured_output(AnalysisReport, method="function_calling"))

In [20]:
report_result = report_chain.invoke(analysis_result["output"])

In [21]:
report_result

AnalysisReport(strengths=['Space Optimization and Accessibility: The modular design of Smart Vertical Farming Pods allows for efficient use of limited urban space. By utilizing rooftops, vacant lots, and indoor areas, the SVFP can bring fresh produce closer to consumers, reducing the need for transportation and making urban farming more accessible to city dwellers.', 'Technological Integration: The incorporation of IoT sensors and a mobile app for monitoring and managing crops enhances user experience and efficiency. Real-time data on environmental conditions and automated care reminders can lead to higher crop yields and better resource management, appealing to both novice and experienced urban farmers.', 'Sustainability and Food Security: SVFP addresses critical issues such as food security and sustainability by providing a local source of fresh, pesticide-free produce. This not only reduces the carbon footprint associated with transporting food but also promotes environmentally frie

In [22]:
report_result.strengths

['Space Optimization and Accessibility: The modular design of Smart Vertical Farming Pods allows for efficient use of limited urban space. By utilizing rooftops, vacant lots, and indoor areas, the SVFP can bring fresh produce closer to consumers, reducing the need for transportation and making urban farming more accessible to city dwellers.',
 'Technological Integration: The incorporation of IoT sensors and a mobile app for monitoring and managing crops enhances user experience and efficiency. Real-time data on environmental conditions and automated care reminders can lead to higher crop yields and better resource management, appealing to both novice and experienced urban farmers.',
 'Sustainability and Food Security: SVFP addresses critical issues such as food security and sustainability by providing a local source of fresh, pesticide-free produce. This not only reduces the carbon footprint associated with transporting food but also promotes environmentally friendly practices in urban

In [23]:
report_result.weaknesses

['Initial Cost and Investment: The upfront cost of purchasing and installing Smart Vertical Farming Pods may be a barrier for some potential users, particularly individuals or small businesses with limited budgets. The need for advanced technology and infrastructure could deter widespread adoption in lower-income urban areas.',
 'Technical Complexity: While the technology aims to simplify urban farming, the reliance on IoT and app-based management may pose challenges for users who are not tech-savvy. This could limit the target market to those comfortable with technology, potentially excluding a segment of the population interested in urban farming.',
 'Maintenance and Reliability: The success of SVFP depends on the reliability of its technology and systems. Any malfunction in the hydroponic systems, sensors, or app could lead to crop failure or reduced yields. Additionally, ongoing maintenance and support will be necessary to ensure optimal performance, which could increase operationa

## E2E Chain

In [24]:
from langchain_core.runnables import RunnableLambda

e2e_chain = (
    idea_chain
    | RunnableLambda(lambda x: {"idea": x})
    | analysis_chain
    | report_chain
)

In [26]:
report = e2e_chain.invoke("interior design")

In [27]:
report

AnalysisReport(strengths=['Immersive Experience: VRIDE offers a unique and engaging way for clients to visualize their interior design choices in a fully immersive 3D environment, leading to higher customer satisfaction.', 'Personalization through AI: The integration of AI to suggest design styles, color palettes, and furniture arrangements tailored to individual preferences enhances the personalization aspect of the service, attracting a wider range of clients.', 'Streamlined Purchasing Process: By incorporating a marketplace for furniture and decor items, VRIDE simplifies the transition from design to purchase, increasing sales and improving the overall customer experience.'], weaknesses=['High Initial Investment: Developing a sophisticated VR platform and maintaining the technology can require significant upfront investment, posing a financial risk for startups or small businesses.', 'Technology Accessibility: Not all potential clients may have access to VR headsets or the necessary

In [28]:
report.strengths

['Immersive Experience: VRIDE offers a unique and engaging way for clients to visualize their interior design choices in a fully immersive 3D environment, leading to higher customer satisfaction.',
 'Personalization through AI: The integration of AI to suggest design styles, color palettes, and furniture arrangements tailored to individual preferences enhances the personalization aspect of the service, attracting a wider range of clients.',
 'Streamlined Purchasing Process: By incorporating a marketplace for furniture and decor items, VRIDE simplifies the transition from design to purchase, increasing sales and improving the overall customer experience.']

In [29]:
report.weaknesses

['High Initial Investment: Developing a sophisticated VR platform and maintaining the technology can require significant upfront investment, posing a financial risk for startups or small businesses.',
 'Technology Accessibility: Not all potential clients may have access to VR headsets or the necessary technology to fully utilize the platform, limiting the target market and excluding less tech-savvy customers.',
 "Dependence on Accurate Measurements: The platform's success relies heavily on clients providing accurate dimensions and photos of their spaces, as inaccuracies could lead to unsatisfactory designs and damage the brand's reputation."]

In [30]:
e2e_chain.get_graph().print_ascii()

              +-------------+            
              | PromptInput |            
              +-------------+            
                      *                  
                      *                  
                      *                  
             +----------------+          
             | PromptTemplate |          
             +----------------+          
                      *                  
                      *                  
                      *                  
               +------------+            
               | ChatOpenAI |            
               +------------+            
                      *                  
                      *                  
                      *                  
       +---------------------------+     
       | Parallel<output,log>Input |     
       +---------------------------+     
               ***         ***           
              *               *          
            **                 ** 